In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
sns.set_theme(style="whitegrid", font_scale=1.0)

In [3]:
df = pd.read_csv('Car_Price_Cleaned.csv')

Optimal Alpha (penalty): 0.3511
Test R^2 Score: 0.8806
Test RMSE (Log Scale): 0.1800


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# 1. Feature Set & Target Definition
num_features = ['curbweight', 'enginesize', 'horsepower', 'citympg']
cat_features = ['drivewheel', 'carbody', 'enginelocation']

X = df[num_features + cat_features]
y_log = np.log1p(df['price'])  # Log transform price to handle skewness
y_actual = df['price']

# 2. Train-Test Split (Splitting both log target and actual dollar target)
X_train, X_test, y_train_log, y_test_log, y_train_act, y_test_act = train_test_split(
    X, y_log, y_actual, test_size=0.2, random_state=42
)

# 3. Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# 4. Fit RidgeCV Model
alphas = np.logspace(-3, 3, 100)
ridge_model = RidgeCV(alphas=alphas)
ridge_model.fit(X_train_prep, y_train_log)

# 5. Predict on Test Set
y_pred_log = ridge_model.predict(X_test_prep)
y_pred_dollar = np.expm1(y_pred_log)  # Exponentiate back to actual dollar values

# 6. Calculate Detailed Performance Metrics
n = X_test.shape[0]          # Number of test observations
p = X_train_prep.shape[1]    # Number of features after One-Hot Encoding

In [7]:
# Metrics on Original Dollar Scale
r2 = r2_score(y_test_act, y_pred_dollar)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae = mean_absolute_error(y_test_act, y_pred_dollar)
rmse = np.sqrt(mean_squared_error(y_test_act, y_pred_dollar))
mape = mean_absolute_percentage_error(y_test_act, y_pred_dollar) * 100

# 7. Display Results
print("=" * 45)
print("       RIDGE REGRESSION MODEL PERFORMANCE     ")
print("=" * 45)
print(f"Optimal Alpha Penalty : {ridge_model.alpha_:.4f}")
print("-" * 45)
print(f"R² Score              : {r2:.4f}")
print(f"Adjusted R² Score     : {adj_r2:.4f}")
print(f"Mean Absolute Error   : ${mae:,.2f}")
print(f"Root Mean Sq. Error   : ${rmse:,.2f}")
print(f"Mean Abs. % Error     : {mape:.2f}%")
print("=" * 45)

       RIDGE REGRESSION MODEL PERFORMANCE     
Optimal Alpha Penalty : 0.3511
---------------------------------------------
R² Score              : 0.8576
Adjusted R² Score     : 0.8036
Mean Absolute Error   : $2,054.84
Root Mean Sq. Error   : $3,352.62
Mean Abs. % Error     : 13.64%
